# Chapter 15: your first week, with code

Section 15.5: build the Kestrel slice (3 brew days, 6 fermenters, 2 packaging lines, about 40 lots) through
bronze, silver and gold, then run the forward trace on one malt lot and time it. Compare the time with the
number you wrote down on Thursday.

In [ ]:
import sys
sys.path.insert(0, "..")
import sqlite3
import time
import pandas as pd
import kestrel as k
import pipeline

k.OUT.mkdir(exist_ok=True)
if pipeline.DB.exists():
    pipeline.DB.unlink()
summary = pipeline.run()
print(f"{summary['files_landed']} files landed in bronze; {len(summary['fingerprint'])} silver and gold tables")

In [ ]:
con = sqlite3.connect(pipeline.DB)
size = pd.read_sql_query("""
    SELECT (SELECT COUNT(DISTINCT date(mash_in_utc, '+5 hours', '+30 minutes')) FROM brew) AS brew_days,
           (SELECT COUNT(DISTINCT vessel_key) FROM vessel_event WHERE vessel_key IN
              (SELECT vessel_key FROM vessel WHERE vessel_type = 'fermenter')) AS fermenters,
           (SELECT COUNT(*) FROM equipment WHERE equipment_type LIKE '%line%') AS packaging_lines,
           (SELECT COUNT(*) FROM lot) AS lots""", con)
print(size.to_string(index=False))

In [ ]:
start = time.perf_counter()
trace = pd.read_sql_query((k.SQL / "lineage_forward.sql").read_text(), con)
seconds = time.perf_counter() - start
pallets = trace[trace.node.str.len() == 18]
print(f"forward trace of M-26-0087: {len(pallets)} pallets at share {pallets.share.round(3).unique()} "
      f"in {seconds * 1000:.1f} ms")
print("Your Thursday number, by messages and spreadsheets: ____ minutes")